# 1. Where the Food Data Comes From

This notebook walks through the public data files behind the opening maps of *Food Out of Reach*:

1. **The source files are intact.** Every dataset matches the digital fingerprint recorded when it was downloaded, and the travel-time file behind the map was built from exactly these files.
2. **The 548 food places.** How MVRPC's directory splits into **45 full-service supermarkets**, **73 emergency food pantries** and **430 other food retailers**.
3. **The 35 USDA low-access tracts.** How the federal food-access map flags 35 urban Census tracts in Montgomery County.

**How to read this notebook:** Each grey box runs a small piece of Python and prints what it found right below it. A box that starts with `assert` is a check: if any number disagrees with what the site publishes, the notebook stops with an error. Each check is written out in words just above it.

In [1]:
import hashlib
import json
from collections import Counter
from pathlib import Path

snapshots_folder = Path('..') / 'snapshots'


def count_of(items):
    return len(items)

## 1. Checking the source files

Every dataset was downloaded once and saved in `snapshots/`, with its SHA-256 digital fingerprint written into `manifest.json` at the moment of download. Changing a single byte of a file changes its fingerprint, so recomputing the fingerprints now confirms that none of the downloaded data has been altered.

Six files were downloaded: five map layers from MVRPC (food pantries, SNAP and WIC retailers, bus route lines, Census block groups and the USDA tracts) and GDRTA's bus timetable feed.

In [2]:
def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


manifest = json.loads((snapshots_folder / 'manifest.json').read_text())
downloaded_files = manifest['layers'] + [manifest['gtfs']]

for downloaded_file in downloaded_files:
    fingerprint_on_disk = sha256_of(snapshots_folder / downloaded_file['file'])
    verdict = 'matches' if fingerprint_on_disk == downloaded_file['sha256'] else 'CHANGED'
    print(f"{downloaded_file['file']:<45} downloaded {downloaded_file['fetched_at'][:10]}   {fingerprint_on_disk[:16]}…  {verdict}")

MC_Pantries@2026-10-03.geojson                downloaded 2026-10-03   60474f379e64b0bb…  matches
SNAP_or_WIC_Acceptance@2026-10-03.geojson     downloaded 2026-10-03   1b7000ef2a3505a0…  matches
MC_Food_Transp@2026-10-03.geojson             downloaded 2026-10-03   f0f88403d123ef0b…  matches
Quantile_Vehicle@2026-10-03.geojson           downloaded 2026-10-03   e6ca20a24c4b3eed…  matches
Low_Income_Low_Access@2026-10-12.geojson      downloaded 2026-10-12   e7632af7eaab7c94…  matches
GDRTA_GTFS@2026-10-03.zip                     downloaded 2026-10-03   a8e0c2b9a5d73407…  matches


**Check:** six files were downloaded, and every one still matches the fingerprint recorded when it was downloaded.

In [3]:
assert count_of(downloaded_files) == 6
assert all(sha256_of(snapshots_folder / downloaded_file['file']) == downloaded_file['sha256'] for downloaded_file in downloaded_files)

We also pre-build three files from raw downloads: the OpenStreetMap street network, the GDRTA bus timetable, and the 2020 Census block housing counts. Each file stores an 8-character fingerprint after `#` in its ID (such as `OSM_streets@2026-10-03#4d561a74`), which we recompute here to confirm none of them has changed.

In [4]:
def fingerprint_of(content, sort_keys=False):
    return hashlib.sha256(json.dumps(content, separators=(',', ':'), sort_keys=sort_keys).encode()).hexdigest()


def short_fingerprint(content, sort_keys=False):
    return fingerprint_of(content, sort_keys)[:8]


street_network = json.loads((snapshots_folder / 'streets@2026-10-03.json').read_text())
bus_timetable = json.loads((snapshots_folder / 'GDRTA_timetable@2026-10-03.json').read_text())
census_blocks = json.loads((snapshots_folder / 'census-blocks@2026-10-06.json').read_text())

built_files = [
    {'name': street_network['snapshotId'], 'recomputed': short_fingerprint({'nodes': street_network['nodes'], 'edges': street_network['edges']})},
    {
        'name': bus_timetable['snapshotId'],
        'recomputed': short_fingerprint({'stops': bus_timetable['stops'], 'patterns': bus_timetable['patterns']}, sort_keys=True),
    },
    {'name': census_blocks['snapshotId'], 'recomputed': short_fingerprint(census_blocks['blocks'])},
]

for built_file in built_files:
    verdict = 'matches' if built_file['name'].endswith('#' + built_file['recomputed']) else 'CHANGED'
    print(f"{built_file['name']:<40} recomputed #{built_file['recomputed']}  {verdict}")

OSM_streets@2026-10-03#4d561a74          recomputed #4d561a74  matches
GDRTA_timetable@2026-10-03#50fc0aed      recomputed #50fc0aed  matches
Census_blocks_2020@2026-10-06#9d5f46d8   recomputed #9d5f46d8  matches


**Check:** all three built files match the fingerprints in their names.

In [5]:
assert all(built_file['name'].endswith('#' + built_file['recomputed']) for built_file in built_files)

Finally, `travel-surface@2026-10-03.json`, the precomputed file that powers the map, lists the fingerprint of every input file used to build it, along with a fingerprint of its own travel-time table.

In [6]:
travel_surface = json.loads((snapshots_folder / 'travel-surface@2026-10-03.json').read_text())

def recorded_name(downloaded_file, service):
    return f"{service}@{downloaded_file['fetched_at'][:10]}#{downloaded_file['sha256'][:8]}"


fingerprints_on_disk = set()
for downloaded_file in manifest['layers']:
    fingerprints_on_disk.add(recorded_name(downloaded_file, downloaded_file['service']))
fingerprints_on_disk.add(recorded_name(manifest['gtfs'], 'GDRTA_GTFS'))
for built_file in built_files:
    fingerprints_on_disk.add(built_file['name'])

for input_name, input_fingerprint in travel_surface['inputs'].items():
    verdict = 'on disk' if input_fingerprint in fingerprints_on_disk else 'MISSING'
    print(f'{input_name:<13} {input_fingerprint:<45} {verdict}')

network       GDRTA_GTFS@2026-10-03#a8e0c2b9                on disk
streets       OSM_streets@2026-10-03#4d561a74               on disk
timetable     GDRTA_timetable@2026-10-03#50fc0aed           on disk
pantries      MC_Pantries@2026-10-03#60474f37               on disk
retailers     SNAP_or_WIC_Acceptance@2026-10-03#1b7000ef    on disk
blockGroups   Quantile_Vehicle@2026-10-03#e6ca20a2          on disk
censusBlocks  Census_blocks_2020@2026-10-06#9d5f46d8        on disk


In [7]:
trip_times_fingerprint = fingerprint_of(travel_surface['cells'])

print(f"trip times recorded  {travel_surface['cellsSha256'][:16]}…")
print(f'trip times on disk   {trip_times_fingerprint[:16]}…')

trip times recorded  3f889c34b09d77b0…
trip times on disk   3f889c34b09d77b0…


**Check:** `travel-surface@2026-10-03.json` was built from the exact source files above, and its travel-time table is intact.

In [8]:
assert all(input_fingerprint in fingerprints_on_disk for input_fingerprint in travel_surface['inputs'].values())
assert trip_times_fingerprint == travel_surface['cellsSha256']

## 2. The 548 places to get food

The Miami Valley Regional Planning Commission (MVRPC) publishes two food layers for Montgomery County: **73** emergency food pantries and **475** stores that accept SNAP or WIC. (We trim stray spaces from MVRPC's text fields as we load them, as the site does.)

In [9]:
def without_extra_spaces(properties):
    return {field: value.strip() if isinstance(value, str) else value for field, value in properties.items()}


pantry_layer = json.loads((snapshots_folder / 'MC_Pantries@2026-10-03.geojson').read_text())
retailer_layer = json.loads((snapshots_folder / 'SNAP_or_WIC_Acceptance@2026-10-03.geojson').read_text())

pantries = [without_extra_spaces(feature['properties']) for feature in pantry_layer['features']]
retailers = [without_extra_spaces(feature['properties']) for feature in retailer_layer['features']]

print(f'{count_of(pantries)} pantries, {count_of(retailers)} SNAP or WIC retailers')

73 pantries, 475 SNAP or WIC retailers


Most of the 475 retailers are gas stations, dollar stores and convenience stores. Only **45** are classified by MVRPC as `FULL SERVICE GROCERY`: supermarkets that stock fresh produce, meat and dairy year-round. Those 45 stores are the supermarkets on the map; the other **430** stay on it as *other food retailers*.

In [10]:
retailers_by_store_type = Counter(retailer['USER_STORE_TYPE'] for retailer in retailers)

for store_type, retailer_count in retailers_by_store_type.most_common():
    print(f'{store_type:<24} {retailer_count:>4}')

GAS STATION               191
DISCOUNT STORE             73
CONVENIENCE STORE          61
FULL SERVICE GROCERY       45
ETHNIC MARKET              30
DRUGSTORE                  24
INTERNATIONAL MARKET       11
SPECIALTY MEAT STORE        9
HEALTH FOOD STORE           7
BULK FOOD STORE             5
FARM MARKET                 4
SMALL SCALE GROCERY         4
WAREHOUSE STORE             3
DEPARTMENT STORE            2
PRODUCE MARKET              1
FARM SUPPLY STORE           1
ETHNIC GROCERY              1
HOME IMPROVEMENT STORE      1
URBAN FARM                  1
BAKERY OUTLET               1


In [11]:
supermarket_store_types = {'FULL SERVICE GROCERY'}

supermarkets = [retailer for retailer in retailers if retailer['USER_STORE_TYPE'] in supermarket_store_types]
other_food_retailers = [retailer for retailer in retailers if retailer['USER_STORE_TYPE'] not in supermarket_store_types]

print(f'{count_of(supermarkets)} supermarkets, {count_of(other_food_retailers)} other food retailers')

45 supermarkets, 430 other food retailers


Here are the first 12 full-service supermarkets, alphabetically:

In [12]:
supermarket_names = sorted(f"{supermarket['USER_ESTABLISHMENT']} ({supermarket['USER_CITY'].title()})" for supermarket in supermarkets)

for supermarket_name in supermarket_names[:12]:
    print(supermarket_name)

ALDI #16 (Huber Heights)
ALDI #42 (Englewood)
ALDI #62 (Vandalia)
ALDI #76 (Dayton)
ALDI #77 (Kettering)
DOROTHY LANE MARKET (Dayton)
DOROTHY LANE MARKET (Oakwood)
DOT'S MARKET AT CROSS POINTE (Centerville)
DOT'S SUPERMARKET (Kettering)
DREXEL FOODTOWN (Dayton)
GEM CITY MARKET COOPERATIVE (Dayton)
GERMANTOWN FRESH MARKET #340 (Germantown)


**Check:** 45 supermarkets, 73 pantries and 430 other food retailers: 548 food places, the number on the map.

In [13]:
food_places = count_of(supermarkets) + count_of(pantries) + count_of(other_food_retailers)

assert count_of(supermarkets) == 45
assert count_of(pantries) == 73
assert count_of(other_food_retailers) == 430
assert food_places == 548

## 3. The 35 USDA low-access tracts

The story opens on the federal government's own food-access map, the USDA Food Access Research Atlas (2019 data on 2010 Census tracts), which MVRPC republishes for the Miami Valley. The USDA flags a low-income tract as **low-access** when at least 500 residents, or a third of its people, live more than **1 mile** from a supermarket in a city, or more than **10 miles** in the country. That is the Atlas field `LILATracts_1And10`.

MVRPC's file covers 168 tracts across five counties:

In [14]:
usda_tract_layer = json.loads((snapshots_folder / 'Low_Income_Low_Access@2026-10-12.geojson').read_text())
atlas_tracts = [without_extra_spaces(feature['properties']) for feature in usda_tract_layer['features']]

tracts_by_county = Counter(f"{atlas_tract['County']} ({atlas_tract['COUNTYFP10']})" for atlas_tract in atlas_tracts)

for county, tract_count in tracts_by_county.most_common():
    print(f"{county:<28} {tract_count:>4} {'tract' if tract_count == 1 else 'tracts'}")

Montgomery (113)              153 tracts
Greene (057)                    9 tracts
Miami (109)                     3 tracts
Warren (165)                    2 tracts
Darke (037)                     1 tract


Keeping Montgomery County (county code `113`) leaves 153 tracts, and 35 of them are flagged as low-income and low-access:

In [15]:
montgomery_county_code = '113'

montgomery_tracts = [atlas_tract for atlas_tract in atlas_tracts if atlas_tract['COUNTYFP10'] == montgomery_county_code]
low_income_low_access_tracts = [montgomery_tract for montgomery_tract in montgomery_tracts if montgomery_tract['LILATracts_1And10'] == 1]
urban_low_income_low_access_tracts = [usda_tract for usda_tract in low_income_low_access_tracts if usda_tract['Urban'] == 1]

print(f'{count_of(montgomery_tracts)} Montgomery County tracts')
print(f'{count_of(low_income_low_access_tracts)} low-income and low-access, {count_of(urban_low_income_low_access_tracts)} of them urban')

153 Montgomery County tracts
35 low-income and low-access, 35 of them urban


All 35 tracts are urban, so the federal standard applies the 1-mile rule to every one of them, which is why the story draws 1-mile circles around the county's supermarkets. Here are the first 10 tracts, with their population and poverty rate:

In [16]:
for usda_tract in sorted(low_income_low_access_tracts, key=lambda tract: tract['GEOID10'])[:10]:
    population, poverty = usda_tract['Pop2010'], usda_tract['PovertyRate']
    print(f"{usda_tract['GEOID10']}  {usda_tract['NAMELSAD10']:<22} population {population:>5,}  poverty {poverty:>5.1f}%")
print(f'… and {count_of(low_income_low_access_tracts) - 10} more')

39113000500  Census Tract 5         population 3,968  poverty  31.3%
39113000600  Census Tract 6         population 1,840  poverty  22.2%
39113000801  Census Tract 8.01      population 3,964  poverty  36.3%
39113000802  Census Tract 8.02      population 2,383  poverty  35.6%
39113000900  Census Tract 9         population 3,133  poverty  37.0%
39113001100  Census Tract 11        population 4,245  poverty  34.3%
39113001200  Census Tract 12        population 1,325  poverty  40.1%
39113001600  Census Tract 16        population 2,072  poverty  20.8%
39113001800  Census Tract 18        population 6,117  poverty  32.0%
39113001900  Census Tract 19        population 4,456  poverty  45.2%
… and 25 more


**Check:** 153 Montgomery County tracts, 35 of them low-income and low-access, and all 35 urban.

In [17]:
assert count_of(montgomery_tracts) == 153
assert count_of(low_income_low_access_tracts) == 35
assert count_of(urban_low_income_low_access_tracts) == 35

## Summary

| Check | Result | Where it appears on the site |
| :--- | :--- | :--- |
| Downloaded files, built files and travel-time inputs | All match | Methodology §5 |
| Full-service supermarkets | **45** | Story step 2, Methodology §1 |
| Emergency food pantries | **73** | Story step 1, Methodology §1 |
| Other SNAP and WIC food retailers | **430** | Story step 1, Methodology §1 |
| Food places on the map | **548** | Story step 1, Methodology §1 and §2 |
| USDA low-income, low-access tracts | **35**, all urban | Story steps 3 and 27 |

The cell below outputs these key figures as JSON, so they can be checked against the numbers the live site shows.

In [18]:
audited_figures = {
    'supermarkets': count_of(supermarkets),
    'pantries': count_of(pantries),
    'otherFoodRetailers': count_of(other_food_retailers),
    'lowIncomeLowAccessTracts': count_of(low_income_low_access_tracts),
}

print(json.dumps(audited_figures, sort_keys=True))

{"lowIncomeLowAccessTracts": 35, "otherFoodRetailers": 430, "pantries": 73, "supermarkets": 45}


To rerun every audit notebook without Jupyter: `python3 notebooks/run_notebooks.py` from the project folder. It needs only Python 3.10 or newer.